In [ ]:
# ==============================================================================
# HÜCRE 1: SİSTEM HAZIRLIĞI VE VERİ YÜKLEME (v5 DINOv2 + LightGlue)
# ==============================================================================
!pip install faiss-cpu folium scikit-learn pillow-heif -q
!pip install git+https://github.com/cvg/LightGlue.git -q

import os, json, zipfile, shutil
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torch
import torch.nn as nn
import torchvision.transforms as T
from google.colab import drive, files
from IPython.display import display, HTML
import folium
import faiss

try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
    print("HEIC desteği aktif.")
except Exception as e:
    print(f"HEIC desteği etkinleşmedi: {e}")

from lightglue import LightGlue, ALIKED, viz2d
from lightglue.utils import load_image as _lg_load_image_orig, rbd
import torch as _torch_for_lightglue

print("Sistem başlatılıyor...")
drive.mount('/content/drive')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# v5 artifact pathleri
DRIVE_DIR = "/content/drive/My Drive/Kirsehir_VPR_DINOv2_v5_Final"
MODEL_NAME = "dinov2_vitb14"
MODEL_POOLING = "cls"
MODEL_PATH = os.path.join(DRIVE_DIR, "dinov2_kirsehir_v5_vitb14_cls_domainaug_best.pth")
FAISS_PATH = os.path.join(DRIVE_DIR, "faiss_index_v5_all.bin")
META_PATH  = os.path.join(DRIVE_DIR, "db_meta_v5_all.json")

DRIVE_ZIP  = "/content/drive/My Drive/kirsehir_data.zip"
LOCAL_ZIP  = "/content/kirsehir_data.zip"
LOCAL_ROOT = "/content/kirsehir_data"

OUTFEATURES = 512
IMG_SIZE = 518
RERANK_TOP_K = 20
LIGHTGLUE_MAX_SIDE = 1024
RESAMPLE_LANCZOS = getattr(Image, "Resampling", Image).LANCZOS

def open_rgb_image(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    return img.convert("RGB")

def load_image(path):
    """LightGlue için EXIF düzeltilmiş ve büyük kenarı sınırlanmış tensor yükleyici."""
    try:
        img_pil = open_rgb_image(path)
        if max(img_pil.size) > LIGHTGLUE_MAX_SIDE:
            img_pil.thumbnail((LIGHTGLUE_MAX_SIDE, LIGHTGLUE_MAX_SIDE), RESAMPLE_LANCZOS)
        img_np = np.array(img_pil)
        return _torch_for_lightglue.from_numpy(img_np).permute(2, 0, 1).float() / 255.0
    except Exception:
        return _lg_load_image_orig(path)

# 1. Veri setini runtime'a çıkarma. db_meta filepath'leri /content/kirsehir_data altını gösterebilir.
if not os.path.exists(LOCAL_ROOT) and os.path.exists(DRIVE_ZIP):
    print("Fotoğraflar Colab ortamına çıkarılıyor...")
    shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)
    with zipfile.ZipFile(LOCAL_ZIP, 'r') as zf:
        zf.extractall(LOCAL_ROOT)
    os.remove(LOCAL_ZIP)
elif os.path.exists(LOCAL_ROOT):
    print(f"Veri seti zaten mevcut: {LOCAL_ROOT}")
else:
    print("Veri seti zip'i bulunamadı; FAISS çalışır ama eşleşme görselleri açılamayabilir.")

entries = os.listdir(LOCAL_ROOT) if os.path.exists(LOCAL_ROOT) else []
if len(entries) == 1 and os.path.isdir(os.path.join(LOCAL_ROOT, entries[0])):
    LOCAL_ROOT = os.path.join(LOCAL_ROOT, entries[0])

# 2. DINOv2 v5 modeli
class GeMPooling(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=1).pow(1.0 / self.p)

class VPRDINOv2(nn.Module):
    def __init__(self, backbone_name=MODEL_NAME, out_dim=OUTFEATURES, pooling=MODEL_POOLING):
        super().__init__()
        self.pooling = pooling.lower()
        self.backbone = torch.hub.load("facebookresearch/dinov2", backbone_name, verbose=False)
        embed_dim = self.backbone.embed_dim

        if self.pooling == "gem":
            self.gem = GeMPooling(p=3)
        elif self.pooling != "cls":
            raise ValueError(f"Desteklenmeyen pooling: {pooling}")

        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.10),
            nn.Linear(embed_dim, out_dim),
        )

    def forward(self, x):
        ret = self.backbone.forward_features(x)
        if self.pooling == "cls":
            x = ret["x_norm_clstoken"]
        else:
            x = self.gem(ret["x_norm_patchtokens"])
        x = self.head(x)
        return nn.functional.normalize(x, p=2, dim=1)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"v5 model bulunamadı. Önce v5 eğitim notebook'unu çalıştır:\n{MODEL_PATH}")
if not os.path.exists(FAISS_PATH) or not os.path.exists(META_PATH):
    raise FileNotFoundError(
        f"v5 all-image FAISS veya metadata bulunamadı:\n{FAISS_PATH}\n{META_PATH}\n"
        "v5 eğitim notebook'undaki all-image FAISS hücresini çalıştır."
    )

dino_model = VPRDINOv2(MODEL_NAME, OUTFEATURES, MODEL_POOLING).to(DEVICE)
state = torch.load(MODEL_PATH, map_location=DEVICE)
if isinstance(state, dict) and "model_state" in state:
    state = state["model_state"]
dino_model.load_state_dict(state, strict=True)
dino_model.eval()

eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Telefon/dış kaynak sorgularında oranı koruyup merkezden kare kırpıyoruz.
phone_query_transform = T.Compose([
    T.Resize(IMG_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. LightGlue modelleri
lg_extractor = ALIKED(max_num_keypoints=1024).eval().to(DEVICE)
lg_matcher = LightGlue(features='aliked').eval().to(DEVICE)

# 4. FAISS indeksi
faiss_index = faiss.read_index(FAISS_PATH)
with open(META_PATH, "r", encoding="utf-8") as f:
    db_meta = json.load(f)

# Global değişkenler
GLOBAL_QUERY_PATH = None
GLOBAL_DINO_TOP20 = []
GLOBAL_BEST_HYBRID = None
GLOBAL_RERANKED_RESULTS = []

print(f"Modeller ve v5 veritabanı yüklendi. Cihaz: {DEVICE}")
print(f"Model: {MODEL_NAME} / {MODEL_POOLING}")
print(f"FAISS vektör sayısı: {faiss_index.ntotal}")

In [ ]:
# ==============================================================================
# HÜCRE 2: YÖNTEM 1 - v5 DINOv2 İLE GLOBAL ARAMA
# ==============================================================================
print("Test etmek istediğiniz fotoğrafı yükleyin:")
uploaded = files.upload()

if uploaded:
    GLOBAL_QUERY_PATH = list(uploaded.keys())[0]
    img_pil = open_rgb_image(GLOBAL_QUERY_PATH)

    img_tensor = phone_query_transform(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
        query_emb = dino_model(img_tensor).cpu().numpy().astype(np.float32)
    faiss.normalize_L2(query_emb)

    search_k = min(RERANK_TOP_K, faiss_index.ntotal)
    dists, idxs = faiss_index.search(query_emb, search_k)

    GLOBAL_DINO_TOP20 = []
    for i in range(search_k):
        match = db_meta[int(idxs[0][i])].copy()
        match["dino_score"] = float(dists[0][i])
        match["db_idx"] = int(idxs[0][i])
        GLOBAL_DINO_TOP20.append(match)

    display(HTML("<h3>YÖNTEM 1: v5 DINOv2 SONUÇLARI (TOP-3)</h3>"))
    n_show = min(3, len(GLOBAL_DINO_TOP20))
    fig, axes = plt.subplots(1, n_show + 1, figsize=(4.5 * (n_show + 1), 4))
    if n_show == 0:
        axes = [axes]

    axes[0].imshow(img_pil)
    axes[0].set_title("SORGUNUZ", color="blue", fontweight="bold")
    axes[0].axis("off")

    for i in range(n_show):
        m_img = open_rgb_image(GLOBAL_DINO_TOP20[i]["filepath"])
        axes[i + 1].imshow(m_img)
        axes[i + 1].set_title(
            f"#{i+1}: {GLOBAL_DINO_TOP20[i]['street'][:24]}\nSkor: {GLOBAL_DINO_TOP20[i]['dino_score']:.3f}",
            fontweight="bold"
        )
        axes[i + 1].axis("off")
    plt.tight_layout()
    plt.show()

    best_dino = GLOBAL_DINO_TOP20[0]
    print(f"DINOv2 v5'e göre en olası konum: {best_dino['street']}")
    print(f"Koordinat: {best_dino['lat']:.6f}, {best_dino['lon']:.6f}")
else:
    print("Fotoğraf yüklenmedi.")

In [ ]:
# ==============================================================================
# HÜCRE 3: YÖNTEM 2 - HİBRİT SİSTEM (v5 DINOv2 + LightGlue RE-RANKING)
# ==============================================================================
if not GLOBAL_DINO_TOP20:
    print("Önce Hücre 2'den fotoğraf yükleyip DINOv2 taraması yapmalısınız.")
else:
    print(f"DINOv2'nin bulduğu {len(GLOBAL_DINO_TOP20)} aday LightGlue ile geometrik olarak taranıyor...\n")

    img0 = load_image(GLOBAL_QUERY_PATH).to(DEVICE)
    feats0 = lg_extractor.extract(img0)

    reranked_results = []

    for rank, cand in enumerate(GLOBAL_DINO_TOP20):
        try:
            img1 = load_image(cand["filepath"]).to(DEVICE)
            feats1 = lg_extractor.extract(img1)

            matches01 = lg_matcher({"image0": feats0, "image1": feats1})
            matches_clean = rbd(matches01)
            num_matches = int(matches_clean["matches"].shape[0])

            del img1, feats1, matches01
        except Exception as e:
            num_matches = 0
            print(f"Aday #{rank + 1} LightGlue hatası: {type(e).__name__}: {e}")

        cand_copy = cand.copy()
        cand_copy["num_matches"] = num_matches
        cand_copy["original_rank"] = rank + 1
        reranked_results.append(cand_copy)

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    del img0, feats0
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    reranked_results = sorted(reranked_results, key=lambda x: x["num_matches"], reverse=True)
    GLOBAL_RERANKED_RESULTS = reranked_results
    GLOBAL_BEST_HYBRID = reranked_results[0]

    display(HTML("<h3>YÖNTEM 2: LIGHTGLUE İLE YENİDEN SIRALANMIŞ SONUÇLAR (TOP-3)</h3>"))
    n_show = min(3, len(reranked_results))
    fig, axes = plt.subplots(1, n_show + 1, figsize=(4.5 * (n_show + 1), 4))
    axes[0].imshow(open_rgb_image(GLOBAL_QUERY_PATH))
    axes[0].set_title("SORGUNUZ", color="blue", fontweight="bold")
    axes[0].axis("off")

    for i in range(n_show):
        m = reranked_results[i]
        m_img = open_rgb_image(m["filepath"])
        axes[i + 1].imshow(m_img)
        title_color = "red" if m["original_rank"] > 3 and i == 0 else "green"
        axes[i + 1].set_title(
            f"YENİ #{i+1}: {m['street'][:24]}\nEşleşme: {m['num_matches']} (Eski: {m['original_rank']})",
            fontweight="bold",
            color=title_color
        )
        axes[i + 1].axis("off")
    plt.tight_layout()
    plt.show()

    print(f"Hibrit sisteme göre konum: {GLOBAL_BEST_HYBRID['street']}")
    print(f"Koordinat: {GLOBAL_BEST_HYBRID['lat']:.6f}, {GLOBAL_BEST_HYBRID['lon']:.6f}")
    print("Not: LightGlue ana karar değil; v5 DINO Top-K adaylarını tanısal olarak yeniden sıralar.")

    m_map = folium.Map(location=[GLOBAL_BEST_HYBRID["lat"], GLOBAL_BEST_HYBRID["lon"]], zoom_start=17)
    folium.Marker(
        [GLOBAL_BEST_HYBRID["lat"], GLOBAL_BEST_HYBRID["lon"]],
        popup=f"Hibrit Tahmin: {GLOBAL_BEST_HYBRID['street']}",
        icon=folium.Icon(color="green", icon="ok-sign")
    ).add_to(m_map)
    display(m_map)

In [ ]:
# ==============================================================================
# HÜCRE 4: ALGORİTMA NASIL KARAR VERDİ? (GÖRSEL KANIT)
# ==============================================================================
if not GLOBAL_BEST_HYBRID:
    print("Önce Hücre 3'ü çalıştırmalısınız.")
else:
    print(f"Hibrit sistemin {GLOBAL_BEST_HYBRID['street']} sokağını seçme nedeni:\n")

    img0 = load_image(GLOBAL_QUERY_PATH)
    img1 = load_image(GLOBAL_BEST_HYBRID["filepath"])

    img0_dev = img0.to(DEVICE)
    img1_dev = img1.to(DEVICE)

    feats0 = lg_extractor.extract(img0_dev)
    feats1 = lg_extractor.extract(img1_dev)
    matches01 = lg_matcher({"image0": feats0, "image1": feats1})

    feats0_c, feats1_c, matches01_c = [rbd(x) for x in [feats0, feats1, matches01]]
    kpts0, kpts1, matches = feats0_c["keypoints"], feats1_c["keypoints"], matches01_c["matches"]

    if matches.shape[0] == 0:
        print("LightGlue eşleşme bulamadı; görsel kanıt çizilemeyecek.")
    else:
        m_kpts0, m_kpts1 = kpts0[matches[..., 0]], kpts1[matches[..., 1]]
        viz2d.plot_images([img0, img1])
        viz2d.plot_matches(m_kpts0, m_kpts1, color="lime", lw=1.5)
        viz2d.add_text(0, "SORGUNUZ", fs=14)
        viz2d.add_text(
            1,
            f"VERİTABANI EŞLEŞMESİ\n{GLOBAL_BEST_HYBRID['street']}\n({GLOBAL_BEST_HYBRID['num_matches']} nokta uyuştu)",
            fs=14
        )
        plt.show()